In [4]:
# # BERTić ABSA
#
# Multi-task category detection + category sentiment classification.
#
# ACD:
#   11 independent categories
#   BCEWithLogitsLoss
#
# ACSA:
#   4 sentiments per category
#   CrossEntropyLoss
#
# Best model:
#   lowest validation loss
#
# Thresholds:
#   tuned on validation after training
#
# Test:
#   evaluated once using the frozen best model and thresholds



In [5]:
pip install -q torch transformers scikit-learn tqdm sentencepiece

In [39]:
import json
import copy
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)

from tqdm.auto import tqdm



In [ ]:
# Configuration
def get_config():
    return {
        "input_file": "annotations.json",

        "model_name": "classla/bcms-bertic",

        "model_folder": "weights",

        "model_name_prefix": "bertic_absa",

        "experiment_name": "runs/bertic_absa",

        "max_length": 512,

        "seed": 42,

        "val_size": 0.10,

        "test_size": 0.10,

        "num_epochs": 10,

        "batch_size": 16,

        "learning_rate": 2e-5,

        "weight_decay": 0.01,

        "warmup_ratio": 0.1,

        "dropout": 0.2,

        "gradient_clip": 1.0,

        "early_stopping_patience": 10,

        "num_workers": 0,
    }


config = get_config()

CATEGORIES = [
    "Baterija",
    "Kamera",
    "Ekran",
    "Memorija",
    "Zvučnici",
    "Izgled",
    "Hardver",
    "Softver",
    "Cena",
    "Performanse",
    "Opšta ocena",
]

SENTIMENTS = [
    "Pozitivan",
    "Negativan",
    "Neutralan",
    "Konflikt",
]

CATEGORY_TO_ID = {
    category: i
    for i, category in enumerate(CATEGORIES)
}

SENTIMENT_TO_ID = {
    sentiment: i
    for i, sentiment in enumerate(SENTIMENTS)
}

ID_TO_CATEGORY = {
    i: category
    for category, i in CATEGORY_TO_ID.items()
}

ID_TO_SENTIMENT = {
    i: sentiment
    for sentiment, i in SENTIMENT_TO_ID.items()
}

NUM_CATEGORIES = len(CATEGORIES)
NUM_SENTIMENTS = len(SENTIMENTS)


random.seed(config["seed"])
np.random.seed(config["seed"])
torch.manual_seed(config["seed"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        config["seed"]
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )




Device: cuda
GPU: Tesla T4


In [ ]:
# ## Load and preprocess data


with open(
    config["input_file"],
    "r",
    encoding="utf-8"
) as f:
    raw_data = json.load(f)

print(
    "Loaded:",
    len(raw_data)
)

def preprocess_item(item):

    text = item.get(
        "comment",
        ""
    ).strip()

    if not text:
        return None

    # --------------------------------------------------------
    # Category presence + sentiment
    # --------------------------------------------------------

    category_presence = np.zeros(
        NUM_CATEGORIES,
        dtype=np.float32
    )

    sentiment_labels = np.full(
        NUM_CATEGORIES,
        -1,
        dtype=np.int64
    )

    for aspect in item.get(
        "aspect_terms",
        []
    ):

        category = aspect.get(
            "category"
        )

        polarity = aspect.get(
            "polarity"
        )

        if (
            category not in CATEGORY_TO_ID
            or
            polarity not in SENTIMENT_TO_ID
        ):
            continue

        category_id = CATEGORY_TO_ID[
            category
        ]

        category_presence[
            category_id
        ] = 1.0

        sentiment_labels[
            category_id
        ] = SENTIMENT_TO_ID[
            polarity
        ]

    return {
        "text": text,

        "category_presence":
            category_presence,

        "sentiment_labels":
            sentiment_labels,
    }



examples = []

for item in raw_data:

    example = preprocess_item(item)

    if example is not None:
        examples.append(
            example
        )

print(
    "Usable examples:",
    len(examples)
)





examples = []

for item in raw_data:

    example = preprocess_item(item)

    if example is not None:
        examples.append(
            example
        )

print(
    "Usable examples:",
    len(examples)
)

Loaded: 17551
Usable examples: 17551


In [ ]:
# ## Train / validation / test split
#
# Splitting is done at comment level so the same comment cannot
# appear in multiple splits.


indices = np.arange(
    len(examples)
)

train_indices, temp_indices = (
    train_test_split(
        indices,
        test_size=(
            config["val_size"]
            +
            config["test_size"]
        ),
        random_state=config["seed"],
        shuffle=True,
    )
)

relative_test_size = (
    config["test_size"]
    /
    (
        config["val_size"]
        +
        config["test_size"]
    )
)

val_indices, test_indices = (
    train_test_split(
        temp_indices,
        test_size=relative_test_size,
        random_state=config["seed"],
        shuffle=True,
    )
)

train_examples = [
    examples[i]
    for i in train_indices
]

val_examples = [
    examples[i]
    for i in val_indices
]

test_examples = [
    examples[i]
    for i in test_indices
]

print(
    "Train:",
    len(train_examples)
)

print(
    "Validation:",
    len(val_examples)
)

print(
    "Test:",
    len(test_examples)
)




Train: 14040
Validation: 1755
Test: 1756


In [ ]:

# ## Tokenizer


tokenizer = AutoTokenizer.from_pretrained(
    config["model_name"]
)




In [22]:
# Dataset

class ABSADataset(Dataset):

    def __init__(
        self,
        examples,
        tokenizer,
        max_length
    ):

        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(
            self.examples
        )

    def __getitem__(self, index):

        example = self.examples[index]

        encoding = self.tokenizer(
            example["text"],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )

        return {
            "input_ids": torch.tensor(
                encoding["input_ids"],
                dtype=torch.long
            ),

            "attention_mask": torch.tensor(
                encoding["attention_mask"],
                dtype=torch.long
            ),

            "category_presence": torch.tensor(
                example["category_presence"],
                dtype=torch.float32
            ),

            "sentiment_labels": torch.tensor(
                example["sentiment_labels"],
                dtype=torch.long
            ),

            "text": example["text"],
        }


def collate_batch(batch):

    padded = tokenizer.pad(
        {
            "input_ids": [
                x["input_ids"]
                for x in batch
            ],

            "attention_mask": [
                x["attention_mask"]
                for x in batch
            ],
        },
        padding=True,
        return_tensors="pt",
    )

    return {
        "input_ids":
            padded["input_ids"],

        "attention_mask":
            padded["attention_mask"],

        "category_presence":
            torch.stack([
                x["category_presence"]
                for x in batch
            ]),

        "sentiment_labels":
            torch.stack([
                x["sentiment_labels"]
                for x in batch
            ]),

        "text": [
            x["text"]
            for x in batch
        ],
    }


train_dataset = ABSADataset(
    train_examples,
    tokenizer,
    config["max_length"]
)

val_dataset = ABSADataset(
    val_examples,
    tokenizer,
    config["max_length"]
)

test_dataset = ABSADataset(
    test_examples,
    tokenizer,
    config["max_length"]
)


train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=config["num_workers"],
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"],
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"],
    collate_fn=collate_batch,
)


print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)


# ## Loss weights

train_presence = np.array([
    x["category_presence"]
    for x in train_examples
])

positive_counts = (
    train_presence.sum(axis=0)
)

negative_counts = (
    len(train_examples)
    -
    positive_counts
)

category_pos_weight = np.ones(
    NUM_CATEGORIES,
    dtype=np.float32
)

for i in range(
    NUM_CATEGORIES
):

    if positive_counts[i] > 0:

        category_pos_weight[i] = (
            negative_counts[i]
            /
            positive_counts[i]
        )


print(
    "Category positive weights:"
)

for i, category in (
    ID_TO_CATEGORY.items()
):

    print(
        f"{category:15s}: "
        f"{category_pos_weight[i]:.3f}"
    )


# %%
sentiment_counts = np.zeros(
    NUM_SENTIMENTS,
    dtype=np.float32
)

for example in train_examples:

    for label in (
        example["sentiment_labels"]
    ):

        if label >= 0:
            sentiment_counts[
                label
            ] += 1


sentiment_weights = np.ones(
    NUM_SENTIMENTS,
    dtype=np.float32
)

total_sentiments = (
    sentiment_counts.sum()
)

for i in range(
    NUM_SENTIMENTS
):

    if sentiment_counts[i] > 0:

        sentiment_weights[i] = (
            total_sentiments
            /
            (
                NUM_SENTIMENTS
                *
                sentiment_counts[i]
            )
        )


print(
    "\nSentiment weights:"
)

for i, sentiment in (
    ID_TO_SENTIMENT.items()
):

    print(
        f"{sentiment:12s}: "
        f"{sentiment_weights[i]:.3f}"
    )





Train batches: 878
Validation batches: 110
Test batches: 110
Category positive weights:
Baterija       : 6.897
Kamera         : 11.177
Ekran          : 17.140
Memorija       : 106.176
Zvučnici       : 33.496
Izgled         : 19.924
Hardver        : 11.294
Softver        : 8.117
Cena           : 18.719
Performanse    : 13.748
Opšta ocena    : 4.680

Sentiment weights:
Pozitivan   : 0.466
Negativan   : 0.616
Neutralan   : 7.296
Konfliktan  : 10.475


In [23]:
# BERTić model

class BERTicABSA(nn.Module):

    def __init__(
        self,
        model_name,
        num_categories,
        num_sentiments,
        dropout
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            model_name
        )

        hidden_size = (
            self.encoder.config.hidden_size
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.category_head = nn.Linear(
            hidden_size,
            num_categories
        )

        self.sentiment_head = nn.Linear(
            hidden_size,
            num_categories
            *
            num_sentiments
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = (
            outputs
            .last_hidden_state[:, 0]
        )

        pooled = self.dropout(
            pooled
        )

        category_logits = (
            self.category_head(
                pooled
            )
        )

        sentiment_logits = (
            self.sentiment_head(
                pooled
            )
        )

        sentiment_logits = (
            sentiment_logits.view(
                -1,
                NUM_CATEGORIES,
                NUM_SENTIMENTS
            )
        )

        return {
            "category_logits":
                category_logits,

            "sentiment_logits":
                sentiment_logits,
        }



In [ ]:
model = BERTicABSA(
    config["model_name"],
    NUM_CATEGORIES,
    NUM_SENTIMENTS,
    config["dropout"]
).to(DEVICE)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)


# ## Loss functions


category_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        category_pos_weight,
        dtype=torch.float32,
        device=DEVICE
    )
)

sentiment_loss_fn = nn.CrossEntropyLoss(
    weight=torch.tensor(
        sentiment_weights,
        dtype=torch.float32,
        device=DEVICE
    ),
    ignore_index=-1
)




Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: classla/bcms-bertic
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parameters: 110,069,047


In [ ]:
# Joint loss

def calculate_loss(
    outputs,
    category_targets,
    sentiment_targets
):

    category_loss = category_loss_fn(
        outputs["category_logits"],
        category_targets
    )

    sentiment_logits = (
        outputs["sentiment_logits"]
    )

    valid = (
        sentiment_targets >= 0
    )

    if valid.any():

        sentiment_loss = F.cross_entropy(
            sentiment_logits[valid],
            sentiment_targets[valid],
            weight=sentiment_loss_fn.weight,
        )

    else:

        sentiment_loss = torch.tensor(
            0.0,
            device=DEVICE
        )

    total_loss = (
        category_loss
        +
        sentiment_loss
    )

    return (
        total_loss,
        category_loss,
        sentiment_loss
    )


# ## Optimizer


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"]
)


total_training_steps = (
    len(train_loader)
    *
    config["num_epochs"]
)

warmup_steps = int(
    total_training_steps
    *
    config["warmup_ratio"]
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)




In [ ]:
# ## Train one epoch


def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    writer,
    global_step
):

    model.train()

    total_loss = 0.0
    total_category_loss = 0.0
    total_sentiment_loss = 0.0

    total_examples = 0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for batch in progress:

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        category_targets = batch[
            "category_presence"
        ].to(DEVICE)

        sentiment_targets = batch[
            "sentiment_labels"
        ].to(DEVICE)

        optimizer.zero_grad(
            set_to_none=True
        )

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        (
            loss,
            category_loss,
            sentiment_loss
        ) = calculate_loss(
            outputs,
            category_targets,
            sentiment_targets
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            config["gradient_clip"]
        )

        optimizer.step()
        scheduler.step()

        batch_size = (
            input_ids.size(0)
        )

        total_loss += (
            loss.item()
            *
            batch_size
        )

        total_category_loss += (
            category_loss.item()
            *
            batch_size
        )

        total_sentiment_loss += (
            sentiment_loss.item()
            *
            batch_size
        )

        total_examples += batch_size

        writer.add_scalar(
            "train/step_loss",
            loss.item(),
            global_step
        )

        writer.add_scalar(
            "train/step_category_loss",
            category_loss.item(),
            global_step
        )

        writer.add_scalar(
            "train/step_sentiment_loss",
            sentiment_loss.item(),
            global_step
        )

        global_step += 1

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return {
        "loss":
            total_loss
            /
            total_examples,

        "category_loss":
            total_category_loss
            /
            total_examples,

        "sentiment_loss":
            total_sentiment_loss
            /
            total_examples,

        "global_step":
            global_step,
    }




In [27]:

# ## Validation
#
# This returns both losses and raw predictions so the same validation
# pass can later be used for threshold tuning.


@torch.no_grad()
def run_validation(
    model,
    loader
):

    model.eval()

    total_loss = 0.0
    total_category_loss = 0.0
    total_sentiment_loss = 0.0

    total_examples = 0

    category_probs = []
    category_targets = []

    sentiment_logits = []
    sentiment_targets = []


    for batch in loader:

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        category_target = batch[
            "category_presence"
        ].to(DEVICE)

        sentiment_target = batch[
            "sentiment_labels"
        ].to(DEVICE)


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        (
            loss,
            category_loss,
            sentiment_loss
        ) = calculate_loss(
            outputs,
            category_target,
            sentiment_target
        )


        batch_size = (
            input_ids.size(0)
        )

        total_loss += (
            loss.item()
            *
            batch_size
        )

        total_category_loss += (
            category_loss.item()
            *
            batch_size
        )

        total_sentiment_loss += (
            sentiment_loss.item()
            *
            batch_size
        )

        total_examples += batch_size


        category_probs.append(
            torch.sigmoid(
                outputs["category_logits"]
            ).cpu().numpy()
        )

        category_targets.append(
            category_target.cpu().numpy()
        )

        sentiment_logits.append(
            outputs["sentiment_logits"]
            .cpu()
            .numpy()
        )

        sentiment_targets.append(
            sentiment_target.cpu().numpy()
        )


    return {
        "loss":
            total_loss / total_examples,

        "category_loss":
            total_category_loss
            /
            total_examples,

        "sentiment_loss":
            total_sentiment_loss
            /
            total_examples,

        "category_probs":
            np.concatenate(
                category_probs,
                axis=0
            ),

        "category_targets":
            np.concatenate(
                category_targets,
                axis=0
            ),

        "sentiment_logits":
            np.concatenate(
                sentiment_logits,
                axis=0
            ),

        "sentiment_targets":
            np.concatenate(
                sentiment_targets,
                axis=0
            ),
    }




In [29]:
# Tune category thresholds


def find_best_threshold(
    probabilities,
    targets
):

    best_threshold = 0.5
    best_f1 = -1.0

    for threshold in np.arange(
        0.10,
        0.91,
        0.01
    ):

        predictions = (
            probabilities
            >=
            threshold
        ).astype(int)

        score = f1_score(
            targets,
            predictions,
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score
            best_threshold = (
                float(threshold)
            )

    return (
        best_threshold,
        best_f1
    )


def tune_category_thresholds(results):

    thresholds = np.zeros(
        NUM_CATEGORIES,
        dtype=np.float32
    )

    for category_id in range(
        NUM_CATEGORIES
    ):

        threshold, score = (
            find_best_threshold(
                results["category_probs"][:,category_id],
                results["category_targets"][:,category_id]
            )
        )

        thresholds[
            category_id
        ] = threshold

    return thresholds



In [ ]:
# Training


writer = SummaryWriter(
    config["experiment_name"]
)

best_val_loss = float("inf")

best_state = None

best_epoch = 0

epochs_without_improvement = 0

history = []

global_step = 0


for epoch in range(
    1,
    config["num_epochs"] + 1
):

    print()
    print("=" * 60)

    print(
        f"Epoch "
        f"{epoch}/"
        f"{config['num_epochs']}"
    )

    print("=" * 60)


    start_time = (
        time.perf_counter()
    )


    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        writer,
        global_step
    )

    global_step = (
        train_metrics["global_step"]
    )


    val_metrics = run_validation(
        model,
        val_loader
    )


    epoch_time = (
        time.perf_counter()
        -
        start_time
    )


    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print(
        f"\nTrain loss: "
        f"{train_metrics['loss']:.4f}"
    )

    print(
        f"  category: "
        f"{train_metrics['category_loss']:.4f}"
    )

    print(
        f"  sentiment: "
        f"{train_metrics['sentiment_loss']:.4f}"
    )

    print(
        f"\nVal loss: "
        f"{val_metrics['loss']:.4f}"
    )

    print(
        f"  category: "
        f"{val_metrics['category_loss']:.4f}"
    )

    print(
        f"  sentiment: "
        f"{val_metrics['sentiment_loss']:.4f}"
    )

    print(
        f"\nEpoch time: "
        f"{epoch_time:.2f}s"
    )


    # --------------------------------------------------------
    # TensorBoard
    # --------------------------------------------------------

    writer.add_scalar(
        "train/loss",
        train_metrics["loss"],
        epoch
    )

    writer.add_scalar(
        "train/category_loss",
        train_metrics["category_loss"],
        epoch
    )

    writer.add_scalar(
        "train/sentiment_loss",
        train_metrics["sentiment_loss"],
        epoch
    )

    writer.add_scalar(
        "validation/loss",
        val_metrics["loss"],
        epoch
    )

    writer.add_scalar(
        "validation/category_loss",
        val_metrics["category_loss"],
        epoch
    )

    writer.add_scalar(
        "validation/sentiment_loss",
        val_metrics["sentiment_loss"],
        epoch
    )

    writer.add_scalar(
        "training/epoch_time",
        epoch_time,
        epoch
    )

    writer.add_scalar(
        "training/learning_rate",
        optimizer.param_groups[0]["lr"],
        epoch
    )

    # Save model checkpoint
    output_dir = Path(
      config["model_folder"]
    )


    checkpoint_path = (
        output_dir
        / f"checkpoint_epoch_{epoch}.pt"
    )

    output_dir.mkdir(
      parents=True,
      exist_ok=True
    )

    epoch_thresholds = (
      tune_category_thresholds(
          val_metrics
      )
    )


    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "categories":CATEGORIES,
            "sentiments":SENTIMENTS,
            "category_thresholds": epoch_thresholds,
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_loss": train_metrics["loss"],
            "val_loss": val_metrics["loss"],
        },
        checkpoint_path
    )


    print(f"Saved checkpoint: {checkpoint_path}")


    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    history.append({
        "epoch": epoch,
        "train_loss":
            train_metrics["loss"],
        "train_category_loss":
            train_metrics["category_loss"],
        "train_sentiment_loss":
            train_metrics["sentiment_loss"],
        "val_loss":
            val_metrics["loss"],
        "val_category_loss":
            val_metrics["category_loss"],
        "val_sentiment_loss":
            val_metrics["sentiment_loss"],
        "epoch_time":
            epoch_time,
    })


    # --------------------------------------------------------
    # Best model
    #
    # Selected ONLY by validation loss.
    # --------------------------------------------------------

    if (
        val_metrics["loss"]
        <
        best_val_loss
    ):

        best_val_loss = (
            val_metrics["loss"]
        )

        best_state = copy.deepcopy(
            model.state_dict()
        )

        best_epoch = epoch

        epochs_without_improvement = 0

        print(
            "\nNew best model."
        )

    else:

        epochs_without_improvement += 1

        print(
            f"\nNo improvement "
            f"({epochs_without_improvement}/"
            f"{config['early_stopping_patience']})"
        )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        epochs_without_improvement
        >=
        config["early_stopping_patience"]
    ):

        print(
            "\nEarly stopping."
        )

        break





Epoch 1/10


Training:   0%|          | 0/878 [00:00<?, ?it/s]


Train loss: 1.4697
  category: 0.6087
  sentiment: 0.8610

Val loss: 1.5378
  category: 0.6467
  sentiment: 0.8912

Epoch time: 455.18s
Saved checkpoint: weights/checkpoint_epoch_1.pt

New best model.

Epoch 2/10


Training:   0%|          | 0/878 [00:00<?, ?it/s]


Train loss: 1.2604
  category: 0.5387
  sentiment: 0.7217

Val loss: 1.5189
  category: 0.6313
  sentiment: 0.8876

Epoch time: 457.27s
Saved checkpoint: weights/checkpoint_epoch_2.pt

New best model.

Epoch 3/10


...


Train loss: 0.6611
  category: 0.3488
  sentiment: 0.3123

Val loss: 1.5629
  category: 0.5800
  sentiment: 0.9830

Epoch time: 462.42s
Saved checkpoint: weights/checkpoint_epoch_8.pt

No improvement (6/10)

Epoch 9/10


Training:   0%|          | 0/878 [00:00<?, ?it/s]


Train loss: 0.6532
  category: 0.3472
  sentiment: 0.3059

Val loss: 1.5629
  category: 0.5800
  sentiment: 0.9830

Epoch time: 461.54s
Saved checkpoint: weights/checkpoint_epoch_9.pt

No improvement (7/10)

Epoch 10/10


Training:   0%|          | 0/878 [00:00<?, ?it/s]


Train loss: 0.6517
  category: 0.3487
  sentiment: 0.3030

Val loss: 1.5629
  category: 0.5800
  sentiment: 0.9830

Epoch time: 460.94s
Saved checkpoint: weights/checkpoint_epoch_10.pt

No improvement (8/10)


In [ ]:
# Restore best model

if best_state is not None:

    model.load_state_dict(
        best_state
    )

print()
print(
    f"Best epoch: {best_epoch}"
)

print(
    f"Best validation loss: "
    f"{best_val_loss:.4f}"
)


# ## Save best model



output_dir = Path(
    config["model_folder"]
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

best_model_path = (
    output_dir
    /
    "best_model.pt"
)


torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "categories":
            CATEGORIES,

        "sentiments":
            SENTIMENTS,

        "category_thresholds":
            None,

        "model_name":
            config["model_name"],

        "max_length":
            config["max_length"],

        "best_epoch":
            best_epoch,

        "best_val_loss":
            best_val_loss,
    },
    best_model_path
)


tokenizer.save_pretrained(
    output_dir
)


print(
    "Saved:",
    best_model_path
)



writer.close()


# ## Final validation pass
#
# This is the first point where we tune category thresholds.
#
# The model is already fixed at the best validation-loss checkpoint.


val_results = run_validation(
    model,
    val_loader
)

print(
    "Validation loss:",
    val_results["loss"]
)

# %%
category_thresholds = np.zeros(
    NUM_CATEGORIES,
    dtype=np.float32
)

print(
    "Validation thresholds:"
)

for category_id, category in (
    ID_TO_CATEGORY.items()
):

    threshold, score = (
        find_best_threshold(val_results["category_probs"][:, category_id],
            val_results["category_targets"][:, category_id]
        )
    )

    category_thresholds[
        category_id
    ] = threshold

    print(
        f"{category:15s} "
        f"threshold={threshold:.2f} "
        f"F1={score:.4f}"
    )


Best epoch: 2
Best validation loss: 1.5189
Saved: weights/best_model.pt
Validation loss: 1.5189441566793327
Validation thresholds:
Baterija        threshold=0.74 F1=0.6043
Kamera          threshold=0.83 F1=0.6264
Ekran           threshold=0.81 F1=0.4851
Memorija        threshold=0.86 F1=0.2889
Zvučnici        threshold=0.90 F1=0.5111
Izgled          threshold=0.84 F1=0.5677
Hardver         threshold=0.77 F1=0.4757
Softver         threshold=0.69 F1=0.4860
Cena            threshold=0.87 F1=0.6029
Performanse     threshold=0.88 F1=0.5526
Opšta ocena     threshold=0.64 F1=0.7592


In [ ]:
# Evaluate category detection

def evaluate_categories(
    results,
    thresholds
):

    probabilities = results[
        "category_probs"
    ]

    targets = results[
        "category_targets"
    ]

    predictions = (
        probabilities
        >=
        thresholds
    ).astype(int)


    # --------------------------------------------------------
    # Per-category metrics
    # --------------------------------------------------------

    metrics = []

    for category_id, category in (
        ID_TO_CATEGORY.items()
    ):

        y_true = targets[
            :,
            category_id
        ]

        y_pred = predictions[
            :,
            category_id
        ]

        metrics.append({
            "category":
                category,

            "accuracy":
                accuracy_score(
                    y_true,
                    y_pred
                ),

            "precision":
                precision_score(
                    y_true,
                    y_pred,
                    zero_division=0
                ),

            "recall":
                recall_score(
                    y_true,
                    y_pred,
                    zero_division=0
                ),

            "f1":
                f1_score(
                    y_true,
                    y_pred,
                    zero_division=0
                ),
        })


    # --------------------------------------------------------
    # Flatten multilabel predictions
    #
    # Each category prediction becomes a binary decision.
    # --------------------------------------------------------

    y_true = targets.reshape(-1)
    y_pred = predictions.reshape(-1)


    # --------------------------------------------------------
    # Overall metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    macro_precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    micro_precision = precision_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    micro_recall = recall_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    micro_f1 = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )


    return {
        "predictions":
            predictions,

        "per_category":
            metrics,

        "accuracy":
            accuracy,

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "micro_precision":
            micro_precision,

        "micro_recall":
            micro_recall,

        "micro_f1":
            micro_f1,

        "weighted_f1":
            weighted_f1,
    }




val_category_metrics = evaluate_categories(
    val_results,
    category_thresholds
)

print(
    f"Category accuracy: "
    f"{val_category_metrics['accuracy']:.4f}"
)

print(
    f"Category macro-P: "
    f"{val_category_metrics['macro_precision']:.4f}"
)

print(
    f"Category macro-R: "
    f"{val_category_metrics['macro_recall']:.4f}"
)

print(
    f"Category macro-F1: "
    f"{val_category_metrics['macro_f1']:.4f}"
)

print(
    f"Category micro-P: "
    f"{val_category_metrics['micro_precision']:.4f}"
)

print(
    f"Category micro-R: "
    f"{val_category_metrics['micro_recall']:.4f}"
)

print(
    f"Category micro-F1: "
    f"{val_category_metrics['micro_f1']:.4f}"
)

print(
    f"Category weighted-F1: "
    f"{val_category_metrics['weighted_f1']:.4f}"
)

print()

for metric in val_category_metrics["per_category"]:

    print(
        f"{metric['category']:15s} "
        f"P={metric['precision']:.4f} "
        f"R={metric['recall']:.4f} "
        f"F1={metric['f1']:.4f}"
    )

Category accuracy: 0.9230
Category macro-P: 0.7313
Category macro-R: 0.8286
Category macro-F1: 0.7686
Category micro-P: 0.9230
Category micro-R: 0.9230
Category micro-F1: 0.9230
Category weighted-F1: 0.9296

Baterija        P=0.5226 R=0.7165 F1=0.6043
Kamera          P=0.6434 R=0.6103 F1=0.6264
Ekran           P=0.4222 R=0.5700 F1=0.4851
Memorija        P=0.1970 R=0.5417 F1=0.2889
Zvučnici        P=0.4894 R=0.5349 F1=0.5111
Izgled          P=0.4710 R=0.7143 F1=0.5677
Hardver         P=0.3676 R=0.6739 F1=0.4757
Softver         P=0.3630 R=0.7350 F1=0.4860
Cena            P=0.4961 R=0.7683 F1=0.6029
Performanse     P=0.5625 R=0.5431 F1=0.5526
Opšta ocena     P=0.6473 R=0.9178 F1=0.7592


In [ ]:
# Evaluate sentiment
#
# Oracle:
#   Assumes the category was detected correctly.
#
# End-to-end:
#   Sentiment is evaluated only when the model's
#   category probability exceeds the learned threshold.

def evaluate_sentiment(
    results,
    thresholds
):

    category_probs = results[
        "category_probs"
    ]

    category_targets = results[
        "category_targets"
    ]

    sentiment_logits = results[
        "sentiment_logits"
    ]

    sentiment_targets = results[
        "sentiment_targets"
    ]

    predicted_sentiments = np.argmax(
        sentiment_logits,
        axis=2
    )


    all_gold_oracle = []
    all_pred_oracle = []

    all_gold_e2e = []
    all_pred_e2e = []

    per_category = []


    # --------------------------------------------------------
    # Per-category evaluation
    # --------------------------------------------------------

    for category_id, category in (
        ID_TO_CATEGORY.items()
    ):

        valid = (
            category_targets[
                :,
                category_id
            ]
            == 1
        ) & (
            sentiment_targets[
                :,
                category_id
            ]
            >= 0
        )


        if not valid.any():
            continue


        gold = sentiment_targets[
            valid,
            category_id
        ]

        predicted = predicted_sentiments[
            valid,
            category_id
        ]


        # ----------------------------------------------------
        # Oracle
        # ----------------------------------------------------

        all_gold_oracle.extend(
            gold.tolist()
        )

        all_pred_oracle.extend(
            predicted.tolist()
        )


        oracle_accuracy = accuracy_score(
            gold,
            predicted
        )

        oracle_precision = precision_score(
            gold,
            predicted,
            average="macro",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )

        oracle_recall = recall_score(
            gold,
            predicted,
            average="macro",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )

        oracle_f1 = f1_score(
            gold,
            predicted,
            average="macro",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )


        # ----------------------------------------------------
        # End-to-end
        # ----------------------------------------------------

        emitted = (
            category_probs[
                valid,
                category_id
            ]
            >=
            thresholds[
                category_id
            ]
        )


        if emitted.any():

            gold_e2e = gold[
                emitted
            ]

            pred_e2e = predicted[
                emitted
            ]

            all_gold_e2e.extend(
                gold_e2e.tolist()
            )

            all_pred_e2e.extend(
                pred_e2e.tolist()
            )


            e2e_accuracy = accuracy_score(
                gold_e2e,
                pred_e2e
            )

            e2e_precision = precision_score(
                gold_e2e,
                pred_e2e,
                average="macro",
                labels=np.arange(
                    NUM_SENTIMENTS
                ),
                zero_division=0
            )

            e2e_recall = recall_score(
                gold_e2e,
                pred_e2e,
                average="macro",
                labels=np.arange(
                    NUM_SENTIMENTS
                ),
                zero_division=0
            )

            e2e_f1 = f1_score(
                gold_e2e,
                pred_e2e,
                average="macro",
                labels=np.arange(
                    NUM_SENTIMENTS
                ),
                zero_division=0
            )

            e2e_weighted_f1 = f1_score(
                gold_e2e,
                pred_e2e,
                average="weighted",
                labels=np.arange(
                    NUM_SENTIMENTS
                ),
                zero_division=0
            )

        else:

            e2e_accuracy = 0.0
            e2e_precision = 0.0
            e2e_recall = 0.0
            e2e_f1 = 0.0
            e2e_weighted_f1 = 0.0


        per_category.append({

            "category":
                category,

            "oracle_accuracy":
                oracle_accuracy,

            "oracle_precision":
                oracle_precision,

            "oracle_recall":
                oracle_recall,

            "oracle_f1":
                oracle_f1,

            "e2e_accuracy":
                e2e_accuracy,

            "e2e_precision":
                e2e_precision,

            "e2e_recall":
                e2e_recall,

            "e2e_f1":
                e2e_f1,

            "e2e_weighted_f1":
                e2e_weighted_f1,

            "gold":
                len(gold),

            "emitted":
                int(
                    emitted.sum()
                ),
        })


    # --------------------------------------------------------
    # Overall Oracle metrics
    # --------------------------------------------------------

    oracle_accuracy = accuracy_score(
        all_gold_oracle,
        all_pred_oracle
    )

    oracle_macro_precision = precision_score(
        all_gold_oracle,
        all_pred_oracle,
        average="macro",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )

    oracle_macro_recall = recall_score(
        all_gold_oracle,
        all_pred_oracle,
        average="macro",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )

    oracle_macro_f1 = f1_score(
        all_gold_oracle,
        all_pred_oracle,
        average="macro",
        labels=np.arange(
            NUM_SENTIMENTS
        ),
        zero_division=0
    )

    oracle_micro_precision = precision_score(
        all_gold_oracle,
        all_pred_oracle,
        average="micro",
        zero_division=0
    )

    oracle_micro_recall = recall_score(
        all_gold_oracle,
        all_pred_oracle,
        average="micro",
        zero_division=0
    )

    oracle_micro_f1 = f1_score(
        all_gold_oracle,
        all_pred_oracle,
        average="micro",
        zero_division=0
    )

    oracle_weighted_f1 = f1_score(
        all_gold_oracle,
        all_pred_oracle,
        average="weighted",
        zero_division=0
    )


    # --------------------------------------------------------
    # Overall End-to-End metrics
    # --------------------------------------------------------

    if all_gold_e2e:

        e2e_accuracy = accuracy_score(
            all_gold_e2e,
            all_pred_e2e
        )

        e2e_macro_precision = precision_score(
            all_gold_e2e,
            all_pred_e2e,
            average="macro",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )

        e2e_macro_recall = recall_score(
            all_gold_e2e,
            all_pred_e2e,
            average="macro",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )

        e2e_macro_f1 = f1_score(
            all_gold_e2e,
            all_pred_e2e,
            average="macro",
            labels=np.arange(
                NUM_SENTIMENTS
            ),
            zero_division=0
        )

        e2e_micro_precision = precision_score(
            all_gold_e2e,
            all_pred_e2e,
            average="micro",
            zero_division=0
        )

        e2e_micro_recall = recall_score(
            all_gold_e2e,
            all_pred_e2e,
            average="micro",
            zero_division=0
        )

        e2e_micro_f1 = f1_score(
            all_gold_e2e,
            all_pred_e2e,
            average="micro",
            zero_division=0
        )

        e2e_weighted_f1 = f1_score(
            all_gold_e2e,
            all_pred_e2e,
            average="weighted",
            zero_division=0
        )

    else:

        e2e_accuracy = 0.0
        e2e_macro_precision = 0.0
        e2e_macro_recall = 0.0
        e2e_macro_f1 = 0.0
        e2e_micro_precision = 0.0
        e2e_micro_recall = 0.0
        e2e_micro_f1 = 0.0
        e2e_weighted_f1 = 0.0


    return {

        "oracle_accuracy":
            oracle_accuracy,

        "oracle_macro_precision":
            oracle_macro_precision,

        "oracle_macro_recall":
            oracle_macro_recall,

        "oracle_macro_f1":
            oracle_macro_f1,

        "oracle_micro_precision":
            oracle_micro_precision,

        "oracle_micro_recall":
            oracle_micro_recall,

        "oracle_micro_f1":
            oracle_micro_f1,

        "oracle_weighted_f1":
            oracle_weighted_f1,


        "e2e_accuracy":
            e2e_accuracy,

        "e2e_macro_precision":
            e2e_macro_precision,

        "e2e_macro_recall":
            e2e_macro_recall,

        "e2e_macro_f1":
            e2e_macro_f1,

        "e2e_micro_precision":
            e2e_micro_precision,

        "e2e_micro_recall":
            e2e_micro_recall,

        "e2e_micro_f1":
            e2e_micro_f1,

        "e2e_weighted_f1":
            e2e_weighted_f1,

        "per_category":
            per_category,
    }




# %%
val_sentiment_metrics = (
    evaluate_sentiment(
        val_results,
        category_thresholds
    )
)


#print(
#    f"Oracle sentiment macro-F1: "
#    f"{val_sentiment_metrics['oracle_f1']:.4f}"
#)

#print(
#    f"End-to-end sentiment macro-F1: "
#    f"{val_sentiment_metrics['e2e_f1']:.4f}"
#)


# Save thresholds into best-model checkpoint

print(best_model_path)

checkpoint = torch.load(
    best_model_path,
    map_location=DEVICE,
    weights_only=False
)

checkpoint[
    "category_thresholds"
] = category_thresholds

torch.save(
    checkpoint,
    best_model_path
)

print(
    "Updated checkpoint with thresholds."
)

weights/best_model.pt
Updated checkpoint with thresholds.


In [51]:
# ## Final test evaluation
#
# Test is evaluated only after:
#
#   - model selection
#   - threshold tuning
#
# No test-based tuning is performed.

def run_test(
    model,
    test_loader,
    category_loss_fn,
    sentiment_loss_fn,
    thresholds
):

    results = run_validation(
        model,
        test_loader
    )


    # --------------------------------------------------------
    # Category metrics
    # --------------------------------------------------------

    category_metrics = (
        evaluate_categories(
            results,
            thresholds
        )
    )


    # --------------------------------------------------------
    # Sentiment metrics
    # --------------------------------------------------------

    sentiment_metrics = (
        evaluate_sentiment(
            results,
            thresholds
        )
    )


    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("TEST RESULTS")
    print("=" * 60)


    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    print(
        f"Loss: "
        f"{results['loss']:.4f}"
    )

    print(
        f"Category loss: "
        f"{results['category_loss']:.4f}"
    )

    print(
        f"Sentiment loss: "
        f"{results['sentiment_loss']:.4f}"
    )


    # --------------------------------------------------------
    # Category detection
    # --------------------------------------------------------

    print()
    print("CATEGORY DETECTION")
    print("-" * 60)

    print(
        f"Accuracy:       "
        f"{category_metrics['accuracy']:.4f}"
    )

    print(
        f"Macro precision: "
        f"{category_metrics['macro_precision']:.4f}"
    )

    print(
        f"Macro recall:    "
        f"{category_metrics['macro_recall']:.4f}"
    )

    print(
        f"Macro F1:        "
        f"{category_metrics['macro_f1']:.4f}"
    )

    print(
        f"Micro precision: "
        f"{category_metrics['micro_precision']:.4f}"
    )

    print(
        f"Micro recall:    "
        f"{category_metrics['micro_recall']:.4f}"
    )

    print(
        f"Micro F1:        "
        f"{category_metrics['micro_f1']:.4f}"
    )

    print(
        f"Weighted F1:     "
        f"{category_metrics['weighted_f1']:.4f}"
    )


    # --------------------------------------------------------
    # Sentiment — Oracle
    # --------------------------------------------------------

    print()
    print("SENTIMENT — ORACLE")
    print("-" * 60)

    print(
        f"Accuracy:       "
        f"{sentiment_metrics['oracle_accuracy']:.4f}"
    )

    print(
        f"Macro precision: "
        f"{sentiment_metrics['oracle_macro_precision']:.4f}"
    )

    print(
        f"Macro recall:    "
        f"{sentiment_metrics['oracle_macro_recall']:.4f}"
    )

    print(
        f"Macro F1:        "
        f"{sentiment_metrics['oracle_macro_f1']:.4f}"
    )

    print(
        f"Micro precision: "
        f"{sentiment_metrics['oracle_micro_precision']:.4f}"
    )

    print(
        f"Micro recall:    "
        f"{sentiment_metrics['oracle_micro_recall']:.4f}"
    )

    print(
        f"Micro F1:        "
        f"{sentiment_metrics['oracle_micro_f1']:.4f}"
    )

    print(
        f"Weighted F1:     "
        f"{sentiment_metrics['oracle_weighted_f1']:.4f}"
    )


    # --------------------------------------------------------
    # Sentiment — End-to-end
    # --------------------------------------------------------

    print()
    print("SENTIMENT — END-TO-END")
    print("-" * 60)

    print(
        f"Accuracy:       "
        f"{sentiment_metrics['e2e_accuracy']:.4f}"
    )

    print(
        f"Macro precision: "
        f"{sentiment_metrics['e2e_macro_precision']:.4f}"
    )

    print(
        f"Macro recall:    "
        f"{sentiment_metrics['e2e_macro_recall']:.4f}"
    )

    print(
        f"Macro F1:        "
        f"{sentiment_metrics['e2e_macro_f1']:.4f}"
    )

    print(
        f"Micro precision: "
        f"{sentiment_metrics['e2e_micro_precision']:.4f}"
    )

    print(
        f"Micro recall:    "
        f"{sentiment_metrics['e2e_micro_recall']:.4f}"
    )

    print(
        f"Micro F1:        "
        f"{sentiment_metrics['e2e_micro_f1']:.4f}"
    )

    print(
        f"Weighted F1:     "
        f"{sentiment_metrics['e2e_weighted_f1']:.4f}"
    )


    # --------------------------------------------------------
    # Per-category category detection
    # --------------------------------------------------------

    print()
    print("PER-CATEGORY DETECTION")
    print("-" * 60)

    for metric in (
        category_metrics["per_category"]
    ):

        print(
            f"{metric['category']:15s} "
            f"P={metric['precision']:.4f} "
            f"R={metric['recall']:.4f} "
            f"F1={metric['f1']:.4f}"
        )


    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    return {
        "results":
            results,

        "category_metrics":
            category_metrics,

        "sentiment_metrics":
            sentiment_metrics,
    }


# %%
test_results = run_test(
    model,
    test_loader,
    category_loss_fn,
    sentiment_loss_fn,
    category_thresholds
)



TEST RESULTS
Loss: 1.4487
Category loss: 0.5978
Sentiment loss: 0.8509

CATEGORY DETECTION
------------------------------------------------------------
Accuracy:       0.9177
Macro precision: 0.7257
Macro recall:    0.8302
Macro F1:        0.7645
Micro precision: 0.9177
Micro recall:    0.9177
Micro F1:        0.9177
Weighted F1:     0.9254

SENTIMENT — ORACLE
------------------------------------------------------------
Accuracy:       0.8211
Macro precision: 0.4810
Macro recall:    0.4884
Macro F1:        0.4803
Micro precision: 0.8211
Micro recall:    0.8211
Micro F1:        0.8211
Weighted F1:     0.8378

SENTIMENT — END-TO-END
------------------------------------------------------------
Accuracy:       0.8136
Macro precision: 0.4789
Macro recall:    0.4855
Macro F1:        0.4771
Micro precision: 0.8136
Micro recall:    0.8136
Micro F1:        0.8136
Weighted F1:     0.8334

PER-CATEGORY DETECTION
------------------------------------------------------------
Baterija        P=0.555

In [52]:
@torch.no_grad()
def predict_comment(
    text,
    model,
    tokenizer,
    thresholds
):

    model.eval()

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_length"],
    )

    encoding = {
        key: value.to(DEVICE)
        for key, value in encoding.items()
    }

    outputs = model(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
    )

    category_probs = torch.sigmoid(
        outputs["category_logits"]
    )[0].cpu().numpy()

    sentiment_probs = torch.softmax(
        outputs["sentiment_logits"],
        dim=-1
    )[0].cpu().numpy()

    predictions = []

    for category_id, category in ID_TO_CATEGORY.items():

        probability = float(
            category_probs[category_id]
        )

        threshold = float(
            thresholds[category_id]
        )

        present = probability >= threshold

        result = {
            "category": category,
            "probability": probability,
            "threshold": threshold,
            "present": present,
            "sentiment": None,
            "sentiment_probabilities": None,
        }

        if present:

            sentiment_id = int(
                np.argmax(
                    sentiment_probs[category_id]
                )
            )

            result["sentiment"] = (
                ID_TO_SENTIMENT[sentiment_id]
            )

            result["sentiment_probabilities"] = {
                ID_TO_SENTIMENT[i]: float(
                    sentiment_probs[category_id, i]
                )
                for i in range(NUM_SENTIMENTS)
            }

        predictions.append(result)

    for result in predictions:

      if not result["present"]:
          continue

      print(
          f"{result['category']:15s} "
          f"prob={result['probability']:.3f} "
          f"threshold={result['threshold']:.2f} "
          f"-> {result['sentiment']}"
      )

    return predictions

In [53]:
text = "Kamera je odlična, ali baterija je baš loša. Ekran je sasvim solidan."

predictions = predict_comment(
    text,
    model,
    tokenizer,
    category_thresholds,
)

#print(predictions)

Baterija        prob=0.934 threshold=0.74 -> Konfliktan
Kamera          prob=0.959 threshold=0.83 -> Pozitivan
Ekran           prob=0.932 threshold=0.81 -> Pozitivan
Izgled          prob=0.938 threshold=0.84 -> Pozitivan
Hardver         prob=0.885 threshold=0.77 -> Neutralan
Softver         prob=0.835 threshold=0.69 -> Konfliktan
Performanse     prob=0.952 threshold=0.88 -> Pozitivan
Opšta ocena     prob=0.938 threshold=0.64 -> Pozitivan


In [54]:

for result in predictions:

    status = (
        "PRESENT"
        if result["present"]
        else "absent"
    )

    print(
        f"{result['category']:15s} "
        f"{result['probability']:.3f} "
        f"({result['threshold']:.2f}) "
        f"{status}"
    )

    if result["sentiment"]:

        print(
            f"    sentiment: "
            f"{result['sentiment']}"
        )

Baterija        0.934 (0.74) PRESENT
    sentiment: Konfliktan
Kamera          0.959 (0.83) PRESENT
    sentiment: Pozitivan
Ekran           0.932 (0.81) PRESENT
    sentiment: Pozitivan
Memorija        0.800 (0.86) absent
Zvučnici        0.854 (0.90) absent
Izgled          0.938 (0.84) PRESENT
    sentiment: Pozitivan
Hardver         0.885 (0.77) PRESENT
    sentiment: Neutralan
Softver         0.835 (0.69) PRESENT
    sentiment: Konfliktan
Cena            0.831 (0.87) absent
Performanse     0.952 (0.88) PRESENT
    sentiment: Pozitivan
Opšta ocena     0.938 (0.64) PRESENT
    sentiment: Pozitivan


In [ ]:
from google.colab import files


files.download('weights/best_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
tensorboard --logdir runs

SyntaxError: invalid syntax (2662693183.py, line 1)